# Day 09 · 멀티에이전트 — 받아 쓰고, 찍어내고, 그걸로 짓는다

어제 **하네스를 손으로 만들었다** (스킬 · 훅 · 플러그인). 오늘은 **팀**을 얹는다.

> 어제 손으로 만들어봤기 때문에, 오늘 **찍어낸 걸 열어서 읽을 수 있다.**

| 랩 | 무엇을 | 배우는 것 |
|---|---|---|
| Lab 1 | **Superpowers** — 스트릭트 하네스를 그대로 써본다 | 강제가 주는 것 / 뺏는 것 |
| Lab 2 | **메타 하네스**로 내 멀티에이전트 팀을 찍어낸다 | 팀을 만드는 하네스 |
| Lab 3 | 그 팀으로 **MCP 호스트 완성** | 위임하고 게이트로 검증 |

**어제의 두 산출물이 오늘 자란다**

```
harness/                       mcp-host/
├── skills/     ← 어제          ├── 채팅 UI     ← 어제
├── hooks/      ← 어제          ├── MCP 클라이언트  ← 오늘
├── skills/orchestrate/ ← 오늘  ├── 에이전트 루프   ← 오늘
└── agents/{planner,coder,      ├── 도구 승인 UI   ← 오늘
          reviewer,tester} ←오늘 └── 영속화(Neon)  ← 오늘
```

**판단 3문** — 멀티로 갈지 먼저 가린다.
① 병렬 분해가 가능한가? ② 가치가 **약 15배 토큰**보다 큰가? ③ 사람이 검수 가능한 단위인가?
**하나라도 No → 단일 + Plan Mode.**


## Lab 1 · Superpowers — 스트릭트 하네스를 겪는다

남이 만든 **방법론이 코드로 강제되는** 하네스를 그대로 써본다.

### 1. 설치

```
/plugin marketplace add obra/superpowers-marketplace
/plugin install superpowers@superpowers-marketplace
```

### 2. 어제 앱에 기능 하나를 시킨다

```
/superpowers:brainstorm

"어제 만든 mcp-host 앱에 '대화 삭제' 기능을 추가하고 싶어"
```

그다음 흐름을 **그대로 따라간다**:

```
/superpowers:write-plan     # 구현 계획
/superpowers:execute-plan   # 서브에이전트 병렬 실행
```

### 3. 관찰 — 무엇이 강제되나

- [ ] **brainstorm이 먼저** 온다 — 바로 코드로 안 간다
- [ ] **실패하는 테스트를 먼저** 쓴다 (RED)
- [ ] 그다음 최소 구현으로 통과시킨다 (GREEN)
- [ ] **task마다 새 서브에이전트**가 뜬다 — 각자 깨끗한 컨텍스트
- [ ] 리뷰가 **2단계**로 돈다 (스펙 준수 → 코드 품질)

### 4. 우회를 시도한다 ★

```
테스트 건너뛰고 바로 구현해줘
```

**막히는지 확인한다.** 그게 **Hard Gate**다 — 부탁이 아니라 강제.

> 어제 만든 `guard.mjs` 훅과 같은 원리다. 다만 **막는 대상이 파일이 아니라 절차**다.

### 5. 판정 — 표를 채운다

| | 강제가 준 것 | 강제가 뺏은 것 |
|---|---|---|
| 품질 | | |
| 속도 | | |
| 자유도 | | |

### 원리

**정적(스트릭트) 하네스 = 미리 정해진 팀 구조.** 내가 팀을 설계하지 않아도 프레임워크가 강제한다.
우회가 안 되는 게 **장점이자 단점**이다 — 내 도메인에 안 맞아도 **못 바꾼다**.

> 그래서 다음 랩에서 **내 도메인의 팀을 직접 찍어낸다.**


## Lab 2 · 메타 하네스로 내 멀티에이전트 팀을 찍어낸다

Superpowers는 **남의 방법론**이다. 이제 **내 도메인의 팀**을 만든다.

### 1. 메타 하네스 설치

```
/plugin marketplace add revfactory/harness
/plugin install harness
```

> `harness`는 **하네스를 만드는 하네스**다 — 도메인 한 줄을 주면 **에이전트 팀 + 그들이 쓸 스킬**을 생성한다.
> Codex 쪽 포트는 `meta-harness`.

### 2. 찍어낸다

```
/harness "MCP 호스트 웹앱 개발팀 — Next.js · Prisma · MCP · 에이전트 루프.
          기획 → 병렬 구현 → 읽기전용 리뷰 → 테스트 게이트 순서를 강제한다."
```

생성되는 것:

```
.claude/
├── agents/
│   ├── planner.md     기획·분해
│   ├── coder.md       TDD 구현
│   ├── reviewer.md    읽기전용 감사
│   └── tester.md      검증 게이트
└── skills/
    └── orchestrate/SKILL.md   리드 — 위임만 한다
```

### 3. 해부한다 ★ — 어제 만들어봤으니 읽힌다

```bash
cat .claude/agents/reviewer.md
cat .claude/skills/orchestrate/SKILL.md
```

**확인할 것:**

| 항목 | 왜 보나 |
|---|---|
| `name` · `description` | description이 곧 **트리거**다 |
| **`tools`** | **이게 권한 경계다** — reviewer에 `Write`가 있으면 안 된다 |
| 역할 프롬프트 | "직접 코딩하지 말고 위임만" 이 들어있나 |
| 순서 강제 | 건너뛰기 금지가 명시됐나 |

> 어제 `SKILL.md`를 손으로 써봤기 때문에 **이게 마법이 아니라 마크다운**임을 안다.

### 4. 손본다 — 권한을 좁힌다

```yaml
# .claude/agents/reviewer.md
tools: Read, Grep, Glob     # Write · Bash 제거 — 못 고치게
model: haiku                # 리뷰는 싼 모델로 (모델 라우팅)
```

```yaml
# .claude/agents/coder.md
tools: Read, Write, Edit, Bash
# 역할 프롬프트에: "자기 worktree 안에서만 쓴다"
```

**tools가 곧 경계다.** 부탁은 무시돼도 **권한은 못 뚫는다**.

### 5. 어제 만든 훅을 붙인다

```bash
cp harness/hooks/guard.mjs .claude/hooks/
```

이제 **누가 시키든** `.env`는 못 건드린다 — @coder가 쓰기 권한이 있어도.

### 6. 검증

```bash
claude plugin validate ./harness
```

`/agents` 로 팀이 뜨는지 확인한다.

### 원리

**메타 하네스 = 팀을 그때그때 만들어 쓴다.**

| | 정적 (Superpowers) | 메타 (harness) |
|---|---|---|
| 팀 구조 | **정해져 있다** | **내가 찍어낸다** |
| 도메인 적합 | 범용 | **내 도메인에 맞춘다** |
| 수정 | 어렵다 | **파일을 열어 고친다** |
| 전제 | — | **안이 뭔지 알아야 고친다** ← 어제 한 것 |


## Lab 3 · MCP 호스트 완성 (핵심)

어제 **내 하네스로** 지은 뼈대(Next.js + Neon + 스트리밍 채팅)를, **오늘 찍어낸 팀으로** 완성한다.
지금은 LLM과 대화만 된다. 여기에 **도구 · 루프 · 승인 게이트 · 영속화**를 붙이면 진짜 MCP 호스트다.

> 보일러플레이트가 아니다. **내가 어제 지은 것** 위에 짓는다.

---

### Step 1 · 스펙 (`/harness:spec`)

```
/harness:spec

"어제 만든 mcp-host 앱을 진짜 MCP 호스트로 만든다.
 - 5강에서 만든 MCP 서버(HTTP)에 연결해 도구 목록을 가져온다
 - LLM이 도구를 고르면 실행하고, 결과를 다시 넣어 반복한다 (에이전트 루프)
 - 도구를 실행하기 전에 사용자에게 승인을 묻는다 (거부 가능)
 - 대화·메시지·도구호출을 Neon에 저장한다"
```

→ 빠진 요구사항을 되묻고 `docs/prd.md` 확정.

### Step 2 · 분해 (`/harness:orchestrate`) — 사람 승인 게이트

```
/harness:orchestrate "MCP 호스트 완성"

@planner — 뼈대 + prd → 독립 chunk 4개

  chunk 1 — MCP 클라이언트        lib/mcp.ts
     서버 연결 · list_tools · call_tool

  chunk 2 — 에이전트 루프          app/api/chat/route.ts
     판단 → 도구 호출 → 관찰 → 반복 (7강 루프 그대로)

  chunk 3 — 도구 승인 UI           app/components/ApprovalDialog.tsx
     "이 도구를 실행할까요? [승인 / 거부]"

  chunk 4 — 영속화                 prisma/schema.prisma + 저장 로직
     Conversation · Message · ToolCall · McpServer

사람 승인 대기
```

**여기서 멈춰서 계획을 읽는다.** chunk가 정말 독립인가? 파일이 겹치지 않나?

> 겹치면 병렬이 안 된다 — **판단 3문의 ①**이 여기서 실전으로 걸린다.

### Step 3 · 병렬 구현 — @coder 4명

승인하면 각 @coder가 **자기 worktree**에서 RED→GREEN TDD로 구현한다.

```
chunk 1·2·3·4 → @coder 병렬 (각자 .claude/worktrees/<id>/)
@reviewer — 스펙 준수 + 코드 품질 (읽기전용)
@tester   — 테스트 실행 → 통과/실패
통과한 worktree만 병합
```

`.env` 를 건드리려 하면 **guard.mjs 훅이 exit 2로 차단**한다 — 그게 정상이다.

---

### 나와야 할 핵심 — 판정 기준

**에이전트 루프 (chunk 2)** — 7강에서 짠 그 루프다:

```typescript
// app/api/chat/route.ts
for (let step = 0; step < MAX_STEPS; step++) {
  const res = await llm.chat.completions.create({
    model: MODEL, messages, tools, stream: true,
  });

  // 도구를 안 부르면 → 정상 종료
  if (!toolCalls.length) break;

  for (const tc of toolCalls) {
    // ★ 승인 게이트 — 승인 없으면 실행하지 않는다
    const approved = await waitForApproval(tc);
    if (!approved) {
      messages.push({ role: "tool", tool_call_id: tc.id,
                      content: "사용자가 거부했습니다." });
      continue;
    }
    const out = await mcp.callTool(tc.function.name, JSON.parse(tc.function.arguments));
    messages.push({ role: "tool", tool_call_id: tc.id, content: JSON.stringify(out) });
  }
}
```

**MCP 클라이언트 (chunk 1)** — 5강 서버를 HTTP로 붙인다:

```python
# 5강 서버를 HTTP 로 띄운다
if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000)   # http://127.0.0.1:8000/mcp
```

Next.js 백엔드가 이 URL에 붙어 `list_tools` → OpenAI function 스펙으로 변환한다.
**7강 `to_openai_tools` 를 TypeScript로 옮기는 것**과 같다.

---

### Step 4 · 행동으로 검증 — 읽지 말고 눌러본다

```bash
npm run dev
```

- [ ] **도구 목록**이 화면에 뜬다 (`search_docs` · `add_task` · `set_priority`)
- [ ] 도구를 부르는 질문을 하면 → **승인 다이얼로그**가 뜬다
- [ ] **승인**하면 실행되고 결과가 대화에 붙는다
- [ ] **거부**하면 실행 안 되고, 에이전트가 그 사실을 알고 답한다
- [ ] 멀티스텝 — `"연차 규정 찾아서 할 일로 추가해줘"` 가 **두 번 도구를 부른다**
- [ ] Neon 콘솔에 **대화·도구호출 로그**가 쌓인다
- [ ] `.env` 는 커밋되지 않았다

> **모르는 코드일수록 읽지 말고 클릭 사이클로 확인한다.**

---

### 완성 — 무엇을 만든 건가

| 부품 | 어디서 왔나 |
|---|---|
| 에이전트 루프 | **7강** — 노트북에서 파이썬으로 짠 그 루프 |
| MCP 도구 | **5강** — 내가 만든 서버 |
| 승인 게이트 | **8강** — Replit 사고를 막는 그 원칙 |
| 병렬 구현 팀 | **9강** — 내가 지은 오케스트레이터 |

> **Claude Code를 이틀 쓰고, 그걸 직접 만들었다.**
> 블랙박스를 연 게 아니라 — **블랙박스를 만들었다.**


## 산출물 체크리스트

**받아 쓴다** (Lab 1)
- [ ] Superpowers를 설치해 **강제 흐름을 겪었다** — brainstorm → 실패테스트 → 구현 → 리뷰
- [ ] **task마다 새 서브에이전트**가 뜨는 걸 봤다
- [ ] **우회를 시도했고 막혔다** — 그게 Hard Gate다
- [ ] 강제가 **준 것과 뺏은 것**을 표로 적었다

**찍어낸다** (Lab 2)
- [ ] 메타 하네스로 **내 도메인의 팀**을 생성했다
- [ ] 생성된 `agents/*.md` 를 **열어서 읽었다** (어제 손으로 만들어봤으니)
- [ ] **`tools`가 권한 경계**임을 확인하고 reviewer에서 쓰기 권한을 뺐다
- [ ] 어제 만든 **`guard.mjs` 훅을 붙였다**

**그걸로 짓는다** (Lab 3)
- [ ] `/harness:spec` → prd → **chunk 4개로 분해** → **사람이 승인**
- [ ] @coder들이 **각자 worktree에서 병렬** 구현했다
- [ ] **MCP 도구 목록**이 웹앱에 뜬다 (5강 서버)
- [ ] **에이전트 루프**가 멀티스텝으로 돈다 (7강 루프)
- [ ] **승인 게이트**가 동작한다 — 거부하면 실행 안 된다 (8강 원칙)
- [ ] 대화·도구호출이 **Neon에 저장**된다
- [ ] 코드를 다 읽지 않고 **클릭 사이클로 검증**했다

> 한 줄: **Claude Code를 이틀 쓰고, 그걸 직접 만들었다.**
